# Valuation and sensitivities
Run all cells after installing with the ui extra and dev group. Only public invented demo data is used. The seeded book has 10,000 Term policies; change the product in the setup to explore the other public books. Results use arbitrary currency units.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import plotly.graph_objects as go
import polars as pl

from avk.api import check, run
from avk.demo.generate import generate

folder = Path(mkdtemp(prefix="avk-notebook-"))
generate(folder, "TERM", 10000, 209)
print(f"Invented inputs and results: {folder}")


def execute(mode):
    config = folder / f"{mode}.json"
    check(config)
    return run(config)


def aggregate(result):
    path = next(p for p in result.files if p.name.endswith("_aggregate_output.csv"))
    return pl.read_csv(path)

Step 0 is the internal instant; step 1 is the reported valuation-date row. Held reserves are the book amount in `cashflows_be_reserves_BE`. Per-policy ratios are not book amounts. This chart shows run-off on the best-estimate surviving lives.


In [ ]:
base = aggregate(execute("valuation"))
held = "cashflows_be_reserves_BE"
opening = base.filter(pl.col("Projection_Step") == 1)
assert opening.height == 1
print(opening.select("Projection_Step", held, "MCEV_bel_BE"))
fig = go.Figure(
    go.Scatter(x=base["Projection_Step"].to_list(), y=base[held].to_list(), name="Held net reserve")
)
fig.update_layout(
    title="Term held reserves", xaxis_title="Projection step", yaxis_title="Currency units"
)
fig.show()

The generated run files declare their table list in `Inputs.Tables`. The `Cases` section in the sensitivity run increases mortality by 15% with other inputs fixed. Read the impact table rather than subtracting per-policy ratios. Positive and negative BEL movements follow the exported cashflow sign convention; they are not automatically gains and losses.


In [ ]:
cases = execute("cases")
impact = pl.read_csv(next(p for p in cases.files if p.name == "cases_impact.csv"))
assert impact.height > 0
print(impact)
fig = go.Figure(
    go.Table(
        header=dict(values=impact.columns),
        cells=dict(values=[impact[c].to_list() for c in impact.columns]),
    )
)
fig.update_layout(title="Mortality +15%: valuation-date impact")
fig.show()